# Research agent (Colab)
Run the cells top to bottom. Needs `GEMINI_API_KEY` in Colab Secrets (Notebook access on).
After "Restart session" run cells 1 and 2 again: the files stay, the settings do not.

## 1. Get the code and install packages

In [ ]:
import os

# The code lives on GitHub. Put your repository address here:
REPO_URL = "https://github.com/Yul04ek/scientific-research-agent.git"
FOLDER = "/content/research-agent"

if not os.path.exists(FOLDER):
    if "YOUR-USER" in REPO_URL:                       # no repository yet: upload the zip instead
        from google.colab import files
        import zipfile
        uploaded = files.upload()
        zipfile.ZipFile(next(iter(uploaded))).extractall("/content")
        os.rename("/content/ai-agent-yu", FOLDER)
    else:
        !git clone {REPO_URL} {FOLDER}
%cd {FOLDER}
!pip install -q -r requirements-research.txt

## 2. Configuration (Gemini through the OpenAI-compatible endpoint)

In [ ]:
import os
from google.colab import userdata

os.environ["API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["BASE_URL"] = "https://generativelanguage.googleapis.com/v1beta/openai/"
os.environ["MODEL_NAME"] = "gemini-3.1-flash-lite"   # 500 requests/day on the free tier; overloaded? try gemini-3.5-flash-lite
os.environ["EMBED_BACKEND"] = "local"                # local embeddings, no API quota
os.environ["DATA_DIR"] = "data"                      # corpus folder
os.environ["EVIDENCE_THRESHOLD"] = "0.774"           # from the calibration in step 6 (recompute for a new corpus)
os.environ["MAX_MODEL_REQUESTS"] = "15"              # limit of model calls per question
os.environ["PYDANTIC_AI_NO_BANNER"] = "1"

# quick test (30 s limit)
from openai import OpenAI
client = OpenAI(base_url=os.environ["BASE_URL"], api_key=os.environ["API_KEY"]).with_options(timeout=30, max_retries=0)
reply = client.chat.completions.create(model=os.environ["MODEL_NAME"], messages=[{"role": "user", "content": "Reply with the single word: ready"}])
print(reply.choices[0].message.content)

## 3. Collect and index papers
Downloads PDFs from arXiv and indexes them (a few minutes). Change `TOPIC` for another subject.
A new topic needs a new corpus: delete the `data/` folder first.

In [ ]:
from coding_assistant.capabilities.literature_search import collect_papers, search_arxiv

TOPIC = 'all:"conformal prediction"'

print(search_arxiv(TOPIC, max_results=5))      # first look
print(collect_papers(TOPIC, max_results=12))

## 4. Topic clusters (ML part)

In [ ]:
from coding_assistant.capabilities.corpus_analysis import corpus_overview
print(corpus_overview())

## 5. Ask the agent
`await` works directly in Colab. `@high` / `@low` in a question sets the reasoning effort.

In [ ]:
from rich.console import Console
from rich.markdown import Markdown
from coding_assistant.research_agent import build_agent, ask

console = Console()
agent = build_agent(console)
history = None

async def q(text):
    global history
    result = await ask(agent, text, console, history)
    console.print(Markdown(result.output))
    history = result.all_messages()

await q("What is split conformal prediction and what kind of coverage guarantee does it give?")

In [ ]:
await q("Check with a small simulation: does split conformal prediction with alpha = 0.1 reach about 90% coverage on synthetic linear-regression data? Compare with what the papers claim.")

In [ ]:
await q("What does the corpus say about CRISPR off-target effects in zebrafish?")     # expected: INSUFFICIENT EVIDENCE

## 6. Evaluation

In [ ]:
# 1) test questions from the corpus (about 20 model calls), 2) retrieval scores and threshold (no model)
!N_EVAL_QUESTIONS=20 python -m eval.build_eval_set
!python -m eval.run_eval retrieval
# Put the suggested threshold into EVIDENCE_THRESHOLD in step 2 and run that cell again.

In [ ]:
# Agent evaluation. Run it here (not with !python): the output is shown live.
# The corpus is frozen during the run. Failed questions are recorded, the run goes on.
# Only some questions again:  await run_eval.run_agent_eval(True, only=["q6", "q7"])
from eval import run_eval
report = await run_eval.run_agent_eval(use_judge=True)
print(report)

## 7. Save results
The corpus is in `data/`, the results in `eval/results/`. Colab deletes them when the session ends: download or copy to Drive.

In [ ]:
# download as a zip ...
!zip -qr /content/results.zip eval/results eval/retrieval_set.jsonl
from google.colab import files
files.download("/content/results.zip")

# ... or copy to Drive
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/research_agent_results && cp -r eval/results data/meta.json /content/drive/MyDrive/research_agent_results/